# Block 4 · Clustering & dimensionality reduction
> Data Mining · SGH Warsaw School of Economics (SMMD-ADA)
> Course anchor dataset: retail-credit / PD portfolio

**Business framing.** Same portfolio — but today nobody tells us the answer. We look for *structure*: customer segments (who are our customers, in four sentences?) and compressed representations (PCA). With no labels there is no "correct", only *useful* — so honesty about how strong the structure really is becomes part of the deliverable.

**Learning goals for this lab**
1. Prepare features for distance-based methods (scaling is not optional).
2. Run k-means; choose k with the elbow and silhouette — and read weak signals honestly.
3. Profile and **name** segments, using outcomes held out of the clustering.
4. See what hierarchical clustering and DBSCAN can find that k-means cannot.
5. Run PCA: scree, loadings, and a 2-D shadow of the portfolio.

**How to work.** Run cells top to bottom. Before you trust any result: **Kernel → Restart & Run All.**

In [ ]:
import sys
sys.path.append("../lecture_1")   # shared course data loader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.max_columns', 30)

from finance_data import load_taiwan
df = load_taiwan()

# Clustering features: behaviour + capacity. Protected traits (sex,
# marriage) are EXCLUDED from the vote on purpose - similarity must not
# be defined by them; profiling may still look at outcomes afterwards.
pay_cols = [c for c in df.columns if c.startswith("pay_status_")]
Xc = pd.DataFrame({
    "log_limit":      np.log10(df["limit_bal"]),
    "age":            df["age"],
    "utilisation":    (df["bill_amt_sep"] / df["limit_bal"]).clip(-0.5, 2.0),
    "months_delayed": (df[pay_cols] > 0).sum(axis=1),
    "pay_ratio":      (df["pay_amt_sep"] / df["bill_amt_sep"].clip(lower=1)).clip(0, 2.0),
    "bill_trend":     ((df["bill_amt_sep"] - df["bill_amt_apr"]) / df["limit_bal"]).clip(-2, 2),
})
Xc.describe().round(2)

## 1. Scaling: watch it break first
Distances add up feature gaps — in raw units, income (tens of thousands) drowns age (tens). Cluster once without scaling and once with, and compare what "similar" ends up meaning.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

raw = Xc[["age"]].assign(limit=10 ** Xc["log_limit"]).to_numpy()
scaled_2d = StandardScaler().fit_transform(raw)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, data, title in [(axes[0], raw, "raw units"),
                        (axes[1], scaled_2d, "standardised")]:
    km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(data)
    ax.scatter(data[:, 0], data[:, 1], c=km.labels_, s=6, cmap="tab10", alpha=0.6)
    ax.set_title(title); ax.set_xlabel("age")
axes[0].set_ylabel("limit (NT$)"); axes[1].set_ylabel("limit (SD)")
plt.tight_layout(); plt.show()

Left panel: the "clusters" are limit bands — age never got a vote. **Standardise before any distance-based method.** From here on we work with all six features, scaled:

In [ ]:
scaler = StandardScaler()
Xs = scaler.fit_transform(Xc)
Xs.shape

## 2. K-means and the choice of k

In [ ]:
from sklearn.metrics import silhouette_score

ks = range(2, 11)
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(Xs, km.labels_, sample_size=2000, random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(list(ks), inertia, "-o"); axes[0].set_xlabel("k"); axes[0].set_ylabel("inertia")
axes[0].set_title("elbow")
axes[1].plot(list(ks), sil, "-s", color="#003366"); axes[1].set_xlabel("k")
axes[1].set_ylabel("mean silhouette"); axes[1].set_title("silhouette")
plt.tight_layout(); plt.show()
pd.Series(sil, index=list(ks), name="silhouette").round(3)

**Read this honestly.** Silhouette peaks at **our k=4, at 0.27** — moderate, genuine grouping (Block 1's synthetic portfolio scored 0.12: one connected cloud; same diagnostic, both verdicts honest). The transactor wall and the delinquent tail are real islands, but the groups touch. The report sentence survives either way: state k, the silhouette, and how firm the borders are.

In [ ]:
km4 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Xs)
seg = pd.Series(km4.labels_, name="segment")
seg.value_counts().sort_index()

## 3. Profile and name the segments
Profile = per-segment means of the clustering features **plus** outcomes held *out* of the clustering — this is where the business value shows up. Note `default` was never an input.

In [ ]:
profile = Xc.assign(segment=seg.values).groupby("segment").mean().round(2)
profile["limit_ntd"] = (10 ** profile["log_limit"]).round(-3)
profile["size"] = seg.value_counts().sort_index()
profile["default_rate"] = df.groupby(seg.values)["default"].mean().round(3)
profile[["size", "limit_ntd", "utilisation", "months_delayed",
         "pay_ratio", "bill_trend", "default_rate"]]

Now the naming craft. A name must be earned by the table — short, concrete, memorable. Our reading (yours may differ — argue from the numbers!):

| segment | name | why |
|---|---|---|
| 0 | mainstream revolvers | high limit, moderate utilisation, pays a slice of the bill |
| 1 | transactors | near-zero utilisation, pays the bill **in full** (pay_ratio > 1) |
| 2 | maxed-out borrowers | low limit, utilisation ~0.9, bills still growing |
| 3 | **distressed delinquents** | 4+ months delayed — and **PD 58%**, nearly 4× the portfolio |

This is the classic industry quartet (transactors / revolvers / maxed-out / distressed), rediscovered from scratch by an unsupervised algorithm. The test: a product manager should predict the profile from the name alone — and the honesty clause travels along: *silhouette 0.27, real but touching groups.*

## 4. Hierarchical clustering: the dendrogram
Pairwise distances get expensive — we use a random sample of 60 customers. Height = the distance at which two groups merged; big vertical jumps = well-separated groups.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

rng = np.random.default_rng(42)
idx = rng.choice(len(Xs), 60, replace=False)
Z = linkage(Xs[idx], method="ward")

plt.figure(figsize=(11, 3.6))
dendrogram(Z, no_labels=True, color_threshold=7.0)
plt.axhline(7.0, ls="--", color="red")
plt.ylabel("merge distance (Ward)")
plt.tight_layout(); plt.show()

Watch for the long vertical runs before the final merges — the dendrogram's way of showing the same real-but-touching groups the silhouette reported.

## 5. DBSCAN: density and noise
On shapes k-means cannot see (two crescents) — and with an outlier bucket, which in fraud work is sometimes the whole deliverable.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons

Xm, _ = make_moons(n_samples=600, noise=0.07, random_state=42)
km2 = KMeans(n_clusters=2, n_init=10, random_state=42).fit(Xm)
db = DBSCAN(eps=0.12, min_samples=10).fit(Xm)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].scatter(Xm[:, 0], Xm[:, 1], c=km2.labels_, s=8, cmap="tab10")
axes[0].set_title("k-means (must cut with a line)")
noise = db.labels_ == -1
axes[1].scatter(Xm[~noise, 0], Xm[~noise, 1], c=db.labels_[~noise], s=8, cmap="tab10")
axes[1].scatter(Xm[noise, 0], Xm[noise, 1], marker="x", s=60, color="red",
                label=f"noise ({noise.sum()})")
axes[1].set_title("DBSCAN (density + noise)"); axes[1].legend()
plt.tight_layout(); plt.show()

## 6. PCA: compress, inspect, project

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(Xs)
evr = pca.explained_variance_ratio_
print("explained variance:", evr.round(3))
print("cumulative:        ", evr.cumsum().round(3))

PC1 carries ~35%, and 90% takes five of six components — moderate compression that tracks the correlation structure exactly: the bill-driven features share an axis, age stands alone. PCA compresses redundancy — some redundancy, some discount.

In [ ]:
loadings = pd.DataFrame(pca.components_[:4].T, index=Xc.columns,
                        columns=[f"PC{i+1}" for i in range(4)]).round(2)
loadings

Read PC1's recipe: +utilisation, +bill_trend, −pay_ratio, −limit → a **revolving-stress axis**. A component you can name is a component you can defend.

In [ ]:
proj = PCA(2).fit_transform(Xs)
plt.figure(figsize=(7, 4.2))
for k in range(4):
    m = seg.values == k
    rate = df.loc[m, "default"].mean()
    plt.scatter(proj[m, 0], proj[m, 1], s=8, alpha=0.6,
                label=f"segment {k} (PD {rate:.1%})")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.legend(); plt.tight_layout(); plt.show()

## 7. The sobering check: what would this look like on noise?

In [ ]:
noise_data = np.random.default_rng(42).uniform(0, 1, size=(800, 2))
km_noise = KMeans(n_clusters=4, n_init=10, random_state=42).fit(noise_data)
plt.figure(figsize=(4.6, 3.8))
plt.scatter(noise_data[:, 0], noise_data[:, 1], c=km_noise.labels_, s=8, cmap="tab10")
plt.title("k-means, k=4, on pure uniform noise")
plt.xticks([]); plt.yticks([])
plt.tight_layout(); plt.show()

Four tidy, colourful, publishable clusters — on structureless noise. Every clustering deliverable should survive the question: *what would this slide look like on noise?*

## 8. Exercises
1. Re-run k-means with `random_state=7` and compare segment assignments (`pd.crosstab`). How stable is the segmentation? Then try `n_init=1` — what changes?
2. Cluster on an 80% subsample and cross-tabulate against the full-sample segments (the resampling stability check).
3. Drop `months_delayed` from the feature set and re-profile. Does the "distressed delinquents" story survive, or did one feature dictate it?
4. **Anomaly shortlist:** compute each customer's distance to their nearest centroid, take the top 1%, and profile them against the portfolio. Would you hand this list to fraud investigators?
5. **Soft memberships:** fit `GaussianMixture(n_components=4, random_state=42)` and look at `predict_proba` for ten customers near segment boundaries. What does a 55/45 membership tell you that k-means cannot?
6. **Components as features:** add `PCA(4)` into the Block 3 tournament pipeline (`make_pipeline(StandardScaler(), PCA(4), LogisticRegression())`) and compare CV AUC with the raw-feature baseline. Explain the result via the scree plot.

*Hand-in for the project team: a named segment table (sizes, profiles, default rates) plus one honest paragraph on how strong the structure really is (silhouette, stability).*

**"Done" for today** = your segment names pass the product-manager test, and your honesty paragraph would survive a sceptical reviewer.

In [ ]:
# scratch cell for the exercises